# Load processed data + merged temperature

In [ ]:
from energy_forecast.features.mrfo import manta_ray_foraging_optimization
import pandas as pd
import numpy as np
import json
from pathlib import Path
from energy_forecast.features.calendar import create_calendar_features, add_cyclical_encoding
from energy_forecast.features.lags import create_lag_features, create_rolling_features
from sklearn.feature_selection import mutual_info_regression
from pathlib import Path
import datetime, os

In [3]:
os.chdir("..") 
df = pd.read_csv("data/processed/demand_with_temperature.csv", index_col=0, parse_dates=True)
df

,settlement_period,nd,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa2_flow,britned_flow,east_west_flow,nemo_flow,is_holiday,temperature
settlement_date,,,,,,,,,,,,,,
2009-01-01 00:00:00,1,37910,54,1403,0,0,0,33,0,0,0,0,1,-1.604160
2009-01-01 00:30:00,2,38047,53,1403,0,0,0,157,0,0,0,0,1,-1.539581
2009-01-01 01:00:00,3,37380,53,1403,0,0,0,511,0,0,0,0,1,-1.475002
2009-01-01 01:30:00,4,36426,50,1403,0,0,0,589,0,0,0,0,1,-1.541587
2009-01-01 02:00:00,5,35687,50,1403,0,0,0,851,0,0,0,0,1,-1.608172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-02-08 21:30:00,44,30670,3855,6562,0,15905,0,12,-4,439,-531,999,0,6.090917
2024-02-08 22:00:00,45,28684,3873,6562,0,15905,0,12,-4,141,-531,897,0,6.224113
2024-02-08 22:30:00,46,27147,3890,6562,0,15905,0,112,-4,139,-531,893,0,6.311527


# Correlation Check — Kendall vs. Mutual Information

### Kendall

In [4]:
df_diff = df.diff()
kendall = df.corr(method='kendall', numeric_only = True)
kendall['nd'].sort_values(key=abs, ascending=False)

nd                           1.000000
embedded_solar_capacity     -0.298669
embedded_wind_capacity      -0.296601
settlement_period            0.290754
pump_storage_pumping        -0.289488
temperature                 -0.169384
non_bm_stor                  0.164887
east_west_flow              -0.156767
embedded_wind_generation    -0.093553
nemo_flow                   -0.075256
britned_flow                 0.075245
embedded_solar_generation    0.067016
is_holiday                  -0.055181
ifa2_flow                    0.011212
Name: nd, dtype: float64

### Mutual information

In [5]:
X = df_diff.drop(columns=['nd']).dropna()
y = df_diff.loc[X.index, 'nd']

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
mi_series

embedded_solar_generation    0.218504
pump_storage_pumping         0.156668
temperature                  0.085055
britned_flow                 0.030019
east_west_flow               0.029394
nemo_flow                    0.026962
non_bm_stor                  0.015832
embedded_wind_generation     0.015616
ifa2_flow                    0.015076
settlement_period            0.014933
embedded_solar_capacity      0.004968
is_holiday                   0.002434
embedded_wind_capacity       0.002336
dtype: float64

Comparing the two methods reveals meaningful divergence for several features.
Mutual information ranks `embedded_solar_generation` as the strongest relevant
feature (0.219), a result Kendall correlation almost entirely missed (0.067) —
consistent with solar generation's relationship to demand being confounded with
time-of-day rather than monotonic. This directly supports the decision to retain
solar and wind generation as features, in contrast to the original thesis's
approach of dropping them based on a limited OLS test.

Conversely, `embedded_solar_capacity` and `embedded_wind_capacity`, which showed
the strongest Kendall correlations, drop to near-zero mutual information —
confirming these correlations were likely a trend artifact rather than a genuine
short-term relationship, since capacity changes infrequently rather than daily.

`temperature` (0.085) and `pump_storage_pumping` (0.157) show consistent,
moderate relevance across both methods and are retained with confidence.

`settlement_period` shows low mutual information despite known time-of-day
effects on demand (established via the boxplots and MSTL decomposition earlier).
This is likely a limitation of applying mutual information to differenced
cyclical data rather than evidence that time-of-day is unimportant; time-of-day
will instead be captured via cyclical (sine/cosine) encodings in the feature
engineering step below, rather than relying on this correlation check alone.

**Features retained going forward:** temperature, pump_storage_pumping,
embedded_solar_generation, embedded_wind_generation, is_holiday, and the
interconnector flow columns (weak individually but potentially useful in
combination, to be confirmed via SHAP). `embedded_solar_capacity` and
`embedded_wind_capacity` are dropped, as their correlation appears to be a
trend artifact rather than genuine signal.

In [6]:
df = df.drop(columns=["embedded_solar_capacity", "embedded_wind_capacity"])

# Feature Construction

With the correlation analysis complete, the following cells build the actual
feature set to be used for modelling. Calendar, cyclical, lag, and rolling
features are all implemented as reusable functions in
`src/energy_forecast/features/`, rather than written inline, so they can be
reused consistently across notebooks and in the final training pipeline.

- **Calendar features** (`create_calendar_features`) provide the basic date/time
  components (hour, day of week, month, etc.) established as relevant in the
  earlier EDA (via boxplots and MSTL seasonality). `year` is created but left out
  of the saved feature set: the test period (2021–2024) lies beyond the training
  years, and models can't extrapolate it reliably (trees treat every later year as
  the last one seen). `year` would likely look important when fitting the training
  data, since GB demand fell steadily over 2009–2021 and `year` is a clean proxy for
  that trend, but that reliance does not carry over to unseen later years. The
  current demand level is instead captured by the lag and rolling features, which
  adapt automatically when demand shifts (e.g. the 2022 energy-crisis drop).
- **Cyclical encoding** (`add_cyclical_encoding`) converts the cyclical calendar
  columns (hour, day of week, month) into sine/cosine pairs, avoiding the
  artificial discontinuity of raw integer encoding at wraparound points (e.g.
  hour 23 to hour 0).
- **Lag features** (`create_lag_features`) are built at lags 48 and 336. The
  ACF/PACF analysis in the previous notebook identified lags 1, 2, 48 and 336 as
  carrying independent signal, but this project forecasts **day-ahead**
  (horizon h = 48 half-hours, direct strategy): to predict demand at time t, only
  demand observed up to t − 48 may be used. Lags 1 and 2 would not be available
  a day in advance, so only the lags ≥ 48 are kept.
- **Rolling statistics** (`create_rolling_features`) add 1-day and 1-week rolling
  mean and standard deviation of demand, to capture recent level and volatility
  beyond a single lag point. The windows end 48 steps before the target
  (`shift=HORIZON`), for the same reason.

- **Grid-operation variables** (`non_bm_stor`, `pump_storage_pumping` and the
  interconnector flows) are outcomes of the target half-hour itself, partly driven
  by that half-hour's demand, so they would not be known a day ahead. They are
  replaced by their values 48 steps earlier (e.g. `ifa2_flow_lag_48`), the same
  rule as for the demand lags.

Weather and embedded generation (temperature, embedded wind/solar) are kept as
observed values for the target half-hour. At day-ahead they would in practice be
forecasts, so the models are evaluated under the common "perfect weather forecast"
assumption.

In [ ]:
df = create_calendar_features(df)
df = add_cyclical_encoding(df, "hour", period=24)
df = add_cyclical_encoding(df, "day_of_week", period=7)
df = add_cyclical_encoding(df, "month", period=12)

# Day-ahead direct forecasting: only demand observed up to t - HORIZON is used to predict t
HORIZON = 48
LAGS = [48, 336]
assert min(LAGS) >= HORIZON, "every lag must be available HORIZON steps ahead"

df = create_lag_features(df, lags=LAGS)
df = create_rolling_features(df, windows=[48, 336], shift=HORIZON)

# Grid-operation variables are outcomes of the target half-hour itself and would not be known a
# day ahead, so they are replaced by their values HORIZON steps earlier (e.g. ifa2_flow_lag_48)
GRID_COLUMNS = [
    "non_bm_stor", "pump_storage_pumping",
    "ifa2_flow", "britned_flow", "east_west_flow", "nemo_flow",
]
for column in GRID_COLUMNS:
    df = create_lag_features(df, lags=[HORIZON], column=column)
df = df.drop(columns=GRID_COLUMNS)

In [9]:
df

,settlement_period,nd,embedded_wind_generation,embedded_solar_generation,non_bm_stor,pump_storage_pumping,ifa2_flow,britned_flow,east_west_flow,nemo_flow,is_holiday,temperature,day_of_month,day_of_week,day_name,hour,day_of_year,quarter,month,year,week_of_year,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,nd_lag_1,nd_lag_2,nd_lag_48,nd_lag_336,nd_rolling_mean_48,nd_rolling_std_48,nd_rolling_mean_336,nd_rolling_std_336
settlement_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2009-01-01 00:00:00,1,37910,54,0,0,33,0,0,0,0,1,-1.604160,1,3,Thursday,0,1,1,1,2009,1,0.000000,1.000000,0.433884,-0.900969,0.500000,0.866025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-01 00:30:00,2,38047,53,0,0,157,0,0,0,0,1,-1.539581,1,3,Thursday,0,1,1,1,2009,1,0.000000,1.000000,0.433884,-0.900969,0.500000,0.866025,37910.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-01 01:00:00,3,37380,53,0,0,511,0,0,0,0,1,-1.475002,1,3,Thursday,1,1,1,1,2009,1,0.258819,0.965926,0.433884,-0.900969,0.500000,0.866025,38047.0,37910.0,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-01 01:30:00,4,36426,50,0,0,589,0,0,0,0,1,-1.541587,1,3,Thursday,1,1,1,1,2009,1,0.258819,0.965926,0.433884,-0.900969,0.500000,0.866025,37380.0,38047.0,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-01 02:00:00,5,35687,50,0,0,851,0,0,0,0,1,-1.608172,1,3,Thursday,2,1,1,1,2009,1,0.500000,0.866025,0.433884,-0.900969,0.500000,0.866025,36426.0,37380.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-02-08 21:30:00,44,30670,3855,0,0,12,-4,439,-531,999,0,6.090917,8,3,Thursday,21,39,1,2,2024,6,-0.707107,0.707107,0.433884,-0.900969,0.866025,0.500000,32436.0,34190.0,32837.0,30390.0,33070.250000,5837.977931,29128.080357,6201.444533
2024-02-08 22:00:00,45,28684,3873,0,0,12,-4,141,-531,897,0,6.224113,8,3,Thursday,22,39,1,2,2024,6,-0.500000,0.866025,0.433884,-0.900969,0.866025,0.500000,30670.0,32436.0,31148.0,28382.0,33025.104167,5848.189974,29128.913690,6201.633423
2024-02-08 22:30:00,46,27147,3890,0,0,112,-4,139,-531,893,0,6.311527,8,3,Thursday,22,39,1,2,2024,6,-0.500000,0.866025,0.433884,-0.900969,0.866025,0.500000,28684.0,30670.0,29514.0,26693.0,32973.770833,5875.766135,29129.812500,6201.546733


## Handling Missing Values from Lag/Rolling Features

The lag and rolling features introduce missing values at the start of the series,
where insufficient history exists to compute them (up to 384 steps, i.e. the one-week
window plus the 48-step day-ahead shift). These rows are dropped, representing a negligible loss
given the dataset spans approximately 16 years.

In [8]:
df.isna().sum()

settlement_period              0
nd                             0
embedded_wind_generation       0
embedded_solar_generation      0
non_bm_stor                    0
pump_storage_pumping           0
ifa2_flow                      0
britned_flow                   0
east_west_flow                 0
nemo_flow                      0
is_holiday                     0
temperature                    0
day_of_month                   0
day_of_week                    0
day_name                       0
hour                           0
day_of_year                    0
quarter                        0
month                          0
year                           0
week_of_year                   0
hour_sin                       0
hour_cos                       0
day_of_week_sin                0
day_of_week_cos                0
month_sin                      0
month_cos                      0
nd_lag_1                       1
nd_lag_2                       2
nd_lag_48                     48
nd_lag_336

In [11]:
df = df.dropna()

In [ ]:
# Full candidate feature set (everything except the target and non-numeric columns).
# `year` is also excluded: the test years lie beyond the training range, where tree models
# can't extrapolate it and MLR/LSTM would extrapolate it unreliably. The demand level is
# already tracked by the lag and rolling features.
PROCESSED_DIR = Path("data/processed")

full_features = [col for col in df.columns if col not in ['nd', 'day_name', 'year']]
df_full = df[full_features + ['nd']]
df_full.to_csv(PROCESSED_DIR / "model_ready_features_day_ahead.csv")

In [ ]:
df_full.shape

## Final Feature Set for This Notebook

The full engineered feature set has been saved to
`data/processed/model_ready_features_day_ahead.csv`.

It includes all engineered features (calendar, cyclical encodings, lags, rolling statistics, and retained correlation-check columns), restricted to information available
day-ahead (48 steps before each target).
This is used to train both the XGBoost and LSTM models, and to compute SHAP importance from the trained XGBoost model, since SHAP requires visibility into every candidate feature to assess which ones genuinely matter.

## Next Steps

Feature selection by MRFO is not performed in this notebook. In the next
notebook, MRFO will be run twice — once using XGBoost as the fitness function,
once using LSTM — with matched population size and iteration count, so any
difference in the resulting feature subsets can be attributed to the model
rather than to an uneven search budget. Each model will then be trained on its
own matching MRFO selection (XGBoost on the XGBoost-selected features, LSTM on
the LSTM-selected features), alongside both models trained on the full feature
set above.